<a href="https://colab.research.google.com/github/rist-kobe/HPC-Programming/blob/main/Tuning/sample_code/15_io-format/15_io-format.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

Install GNU Fortran and NVIDIA HPC SDK (optional for C-only examples; can take ~30 min)

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential gfortran curl gnupg

# Optional: install NVIDIA HPC SDK (large download, not needed for C-only examples).
# Uncomment the following lines to install it.
#!curl -fsSL https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
#!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' | sudo tee /etc/apt/sources.list.d/nvhpc.list
#!sudo apt-get update -y
#!sudo apt-get install -y nvhpc-22-7-cuda-multi

import glob, os
nvhpc_bins = sorted(glob.glob('/opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin'), key=lambda p: tuple(int(x) for x in p.split('/Linux_x86_64/')[1].split('/')[0].replace('-', '.').split('.')), reverse=True)
if nvhpc_bins:
    nvhpc_bin = nvhpc_bins[0]
    current_path = os.environ.get('PATH', '')
    if nvhpc_bin not in current_path.split(':'):
        os.environ['PATH'] = nvhpc_bin + (':' + current_path if current_path else '')
    print('Using NVIDIA HPC SDK:', nvhpc_bin)
else:
    print('NVIDIA HPC SDK not found (optional; not needed for C-only examples).')


Clone the repository and change to the `15_io-format` directory.

In [ ]:
%cd /content
!rm -rf HPC-Programming
!git clone https://github.com/rist-kobe/HPC-Programming.git
%cd HPC-Programming/Tuning/sample_code/15_io-format
!ls


# Formatted (text) vs. Binary outputs   
* Author:   Yukihiro Ota (yota@rist.or.jp)
* Last update: 30th Jan., 2024 

## Instruction: Compile
1. Source code is in `src/`. Choose either fortran, c, or C++ 
2. Change directory

In [ ]:
!cd src/fortran  # Fortran
!cd src/c        # C
!cd src/cpp      # C++


3. Make

In [ ]:
!make


The code is successfully compiled by
  * GNU (9.3.1) on x86-64 systems
  * GNU (8.5.0) on x86-64 systems

## Instruction: Run and do analyses
1. Sample scripts are in `tests/`. Choose either fortran, c, or c++.       
2. Change directory

In [ ]:
!cd tests/c # On c


3. Run a job script, `run.sh`.

In [ ]:
# One example
!bash run.sh
# Another example
!chmod 755 run.sh
!./run.sh


4. The results will be summarized in file, `logfile`. 

## Exercise 
1. Compare measured elapsed times between the cases of text output (format=0 in `logfile`) and binary output (format=1 in `logfile`), changing array size along z-axis (=2nd arg. in command-line options).
   * You may think that CPU time should be almost zero when taking binary output. However, timing data does not always show this behavior. Indeed, CPU may perform something, such as processes associated with I/O and speculative execution. Eventually, it is difficult to completely distill I/O costs from elapsed time, and moreover the behaviors of CPU are quite complex in modern processors.
   * In conclusion, a concrete way of knowing costly part is to examine elapsed time (not CPU time) in specific part of a program, inserting timer routine. Alternatively, do function profiling of a program with wrapper functions of I/O routine, to recognize system functions and the standard library functions as user's own functions.